<img src="causal-wizard-logo.png" width="60" align="left" style="margin-right: 14px;" />

# [Causal Wizard](https://causalwizard.app) &mdash; Identification & Estimation (1/2)

This notebook takes the config JSON and data file the [Causal Wizard](https://causalwizard.app)
site generated for you and runs the actual statistical estimation &mdash; the site itself
deliberately doesn't do this (there's no server; this notebook, and your own data, are
the whole computation).

It re-derives everything from scratch &mdash; column types, the treatment/outcome
encoding, the identified causal estimand &mdash; rather than trusting anything the site
already told you, so if something doesn't check out, you'll see it here too.

**Run this from inside the repo's `notebooks/` directory** (e.g.
`cd causal_wizard_app/notebooks && jupyter notebook`) so the local `causalwizard` package
is found automatically. In Colab, the next cell installs it straight from GitHub instead.

**Output:** a `results.json` file (config/data paths embedded in it). Open
**02-results.ipynb (2/2)** next - it finds that file on its own.

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("causalwizard") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "causalwizard @ git+https://github.com/drawlinson/causal_wizard_app.git#subdirectory=notebooks"],
        check=True,
    )

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from causalwizard import (
    config, identification, estimation_cdpo, estimation_pdfe, counterfactuals,
    propensity, refutation, diagnostics, results_schema, display_utils,
)

## 1. Inputs

Point these at the two files the site told you to download/have handy, and where you
want the `results.json` this notebook produces to be written.

In [ ]:
config_path = "study-config.json"
data_path = "data.csv"
results_path = "results.json"

## 2. Load data and re-derive the treatment/outcome columns

`align_columns()` renames the raw file's columns, by position, to match the config's own
names - the site renames a blank CSV header to `column_N` before anything else sees it
(and generally, pandas' own guess at a column's name doesn't always match the site's), so
trusting position rather than pandas' names avoids a `KeyError` on a column whose name
pandas guessed differently.

In [ ]:
cfg = config.load_config(config_path)
raw_df = config.align_columns(pd.read_csv(data_path), cfg)

prepared = config.prepare_dataframe(cfg, raw_df)

print(f"{len(prepared.df)} rows kept ({prepared.dropped_excluded} excluded by treatment spec, "
      f"{prepared.dropped_na} dropped for missing treatment/outcome values).")
print(f"Treatment: {prepared.treatment_col} (continuous={prepared.treatment_is_continuous})")
label = f", class 1 = {prepared.outcome_class1_label}" if prepared.outcome_class1_label else ""
print(f"Outcome: {prepared.outcome_col} ({prepared.outcome_effective_type}{label})")

## 3. Identification

For Causal Diagram + Potential Outcomes (CD+PO), build the DoWhy causal model from your
diagram and ask DoWhy to identify the effect &mdash; this runs the same graph algorithm the
site's own Check step used, but here independently, straight from your diagram. The
adjustment set DoWhy identifies (`covariate_cols` below) is exactly what the chosen model
will condition on - not just "every other column in your data." Identification is purely
structural (it never looks at the data's actual values, only the graph), so it's safe to
run before the train/test split below - the model gets rebuilt on `train_df` alone once
that split happens, so DoWhy's own estimators (propensity/DML/IV/frontdoor) never see the
held-out rows.

For Panel Data + Fixed Effects (PD+FE) there's no DoWhy graph to identify &mdash;
"identification" is just the entity/time/covariate structure you set up on the site.

In [ ]:
method = cfg["question"]["method"]

if method == "cd+po":
    identified = identification.identify(
        identification.build_causal_model(prepared.df, cfg["graph"], prepared.treatment_col, prepared.outcome_col)
    )
    estimand_names = identification.estimand_names(identified)
    print("Identified estimand(s):", estimand_names)
    if not estimand_names:
        raise ValueError(
            "DoWhy could not identify this effect from your causal diagram - there's likely an "
            "unobserved confounder blocking every backdoor path. Revisit the diagram on the site."
        )
    method_key = cfg["identification"]["model"]["key"]
    estimand_type = method_key.split(".", 1)[0]
    covariate_cols = identification.estimand_variables(identified, estimand_type)
    print(f"Adjustment set for {method_key!r}: {covariate_cols}")
else:
    method_key = "panel.fixed_effects"
    estimand_type = None
    panel = cfg["question"]["panelData"]
    covariate_cols = panel["covariates"]
    print(f"Panel data: entity={panel['entity']}, time={panel['time']}, covariates={covariate_cols}")

## 4. Finalize the sample and split off a held-out test set

Drops any row missing a value in a covariate the chosen model will actually use (on top
of the treatment/outcome cleaning already done above) - a row that's fine for a model
using different covariates can still have a gap in the ones *this* model needs. Doing
this now, before the split, means both the training and held-out sets are clean: a
covariate that's NaN only in a held-out row wouldn't crash anything later since
statsmodels quietly predicts NaN for it, but *would* silently poison every aggregate
generalization metric.

In [ ]:
covariate_types = {c: config.resolve_effective_type(cfg, c) for c in covariate_cols}
required_cols = [prepared.treatment_col, prepared.outcome_col] + covariate_cols
if method == "pd+fe":
    required_cols += [cfg["question"]["panelData"]["entity"], cfg["question"]["panelData"]["time"]]

clean_df, dropped_covariate_na = config.dropna_rows(prepared.df, required_cols)
prepared.dropped_na += dropped_covariate_na

train_df, test_df = config.train_test_split(clean_df, cfg["question"]["splitTestPc"])
print(f"Dropped {dropped_covariate_na} more row(s) missing a required covariate "
      f"({prepared.dropped_na} missing-value rows dropped in total).")
print(f"{len(train_df)} training / {len(test_df)} held out for validation.")

## 5. Estimation

Fits the model the site's Check step selected (`identification.model.key` in the config),
and computes the effect (ATE/ATT/ATC, per your chosen target units). Re-identifies on
`train_df` alone first (cheap - identification is graph-only, see above) so that DoWhy's
own estimators are fit only on the training split, never the held-out rows.

In [ ]:
outcome_is_binary = prepared.outcome_effective_type == "categorical"

if method == "cd+po":
    model = identification.build_causal_model(train_df, cfg["graph"], prepared.treatment_col, prepared.outcome_col)
    identified = identification.identify(model)
    est = estimation_cdpo.estimate(
        model, identified, method_key, train_df, prepared.treatment_col, prepared.outcome_col,
        covariate_cols, covariate_types, outcome_is_binary, cfg["question"]["effect"],
    )
    estimand_info = identification.estimand_info(identified, estimand_type)
    regression_summary = est.summary_text
else:
    panel = cfg["question"]["panelData"]
    est = estimation_pdfe.estimate(
        train_df, panel["entity"], panel["time"], prepared.treatment_col, prepared.outcome_col,
        covariate_cols, covariate_types,
    )
    estimand_info = {"expression": None, "assumptions": None}
    regression_summary = str(est.result.summary())

print(f"Effect ({cfg['question']['effect'].upper()}): {est.effect:.4g}")

## 6. Validation / refutation

In [ ]:
if method == "cd+po":
    validation = refutation.run_cdpo_refutation(model, identified, est.dowhy_estimate)
    accept = refutation.cdpo_accept(validation)
else:
    validation = refutation.run_pdfe_validation(est.z_pvalue, est.f_pvalue)
    accept = refutation.pdfe_accept(validation)

display_utils.show(validation, label="Validation")
print("Accept:", accept)

## 7. Counterfactuals and held-out generalization

Every estimator with a do-operator (own linear regression/GLM, Double ML, and PD+FE)
supports the counterfactual table. A held-out generalization *accuracy* check needs a
real fitted "predict Y from X" model though, which only own linear regression/GLM and
PD+FE have - Double ML's do-operator only gives the shift to a counterfactual value, not
an absolute prediction, so it's excluded from generalization (see
`diagnostics.generalization_unavailable_reason`).

In [ ]:
treatment_spec = cfg["question"]["treatmentSpec"]
cf_control = treatment_spec.get("counterfactualLower")
cf_treated = treatment_spec.get("counterfactualUpper")
cf_control = 0.0 if cf_control is None else cf_control
cf_treated = 1.0 if cf_treated is None else cf_treated

cf_table = counterfactuals.counterfactual_table(
    train_df, prepared.treatment_col, est.predict, prepared.treatment_is_continuous, cf_control, cf_treated,
)
train_preds = counterfactuals.train_predictions(train_df, est.predict, cf_control, cf_treated)

estimator_name = est.estimator if method == "cd+po" else "fixed_effects"
if diagnostics.generalization_unavailable_reason(method, estimator_name) is None:
    generalization = counterfactuals.generalization_predictions(
        test_df, est.predict, prepared.treatment_col, prepared.outcome_col
    )
else:
    generalization = None

display_utils.show(cf_table, label="Counterfactual table")
print("Generalization sample count:", None if generalization is None else len(generalization["actual"]))

## 8. Propensity diagnostics

Only meaningful for the propensity-based CD+PO estimators.

In [ ]:
propensity_analysis = None
propensity_estimators = ("propensity_score_weighting", "propensity_score_matching", "propensity_score_stratification")
if method == "cd+po" and est.estimator in propensity_estimators:
    ps = propensity.extract_propensity_scores(est.dowhy_estimate)
    treatment_binary = train_df[prepared.treatment_col].to_numpy()
    distribution = propensity.positivity_distribution(ps, treatment_binary)
    balance = propensity.covariate_balance(train_df, covariate_cols, covariate_types, treatment_binary, ps)
    propensity_analysis = {"distribution": distribution, "covariate_balance": balance}
    print("Positivity - rows in the unreliable tail bins:", distribution["distribution_bad"])
    display_utils.show(balance, label="Covariate balance")
else:
    print("Not a propensity-based estimator - no positivity/balance diagnostics.")

## 9. Sample counts, contingency table, modelling statements

In [ ]:
contingency = diagnostics.contingency_table(
    train_df, prepared.treatment_col, prepared.outcome_col, outcome_is_binary, prepared.outcome_class1_label,
)
sample_counts = contingency["total"]

modelling_statements = diagnostics.modelling_statements(
    method, prepared.treatment_col, prepared.outcome_col, prepared.treatment_is_continuous,
    prepared.outcome_effective_type, prepared.outcome_class1_label, sample_counts,
    estimand_type, estimand_info["expression"], est.estimator if method == "cd+po" else "fixed_effects",
    cfg["question"]["panelData"] if method == "pd+fe" else None,
)
display_utils.show(sample_counts, label="Sample counts")
display_utils.show(modelling_statements, label="Modelling statements")

## 10. Save `results.json`

In [ ]:
results = results_schema.build_results(
    config_path=config_path, data_path=data_path,
    method=method, treatment_col=prepared.treatment_col, outcome_col=prepared.outcome_col,
    treatment_is_continuous=prepared.treatment_is_continuous, outcome_effective_type=prepared.outcome_effective_type,
    outcome_class1_label=prepared.outcome_class1_label, target_units=cfg["question"]["effect"], effect=est.effect,
    estimand_type=estimand_type,
    estimand_expression=estimand_info["expression"],
    estimator_name=est.estimator if method == "cd+po" else "fixed_effects",
    method_key=method_key, assumptions=estimand_info["assumptions"], validation=validation, accept=accept,
    counterfactuals=cf_table, counterfactual_control_value=cf_control, counterfactual_treated_value=cf_treated,
    generalization=generalization, contingency=contingency, sample_counts=sample_counts,
    propensity_analysis=propensity_analysis, modelling_statements=modelling_statements,
    panel_data=cfg["question"]["panelData"] if method == "pd+fe" else None,
    dropped_excluded=prepared.dropped_excluded, dropped_na=prepared.dropped_na,
    train_predictions=train_preds, estimand_variables=covariate_cols if method == "cd+po" else [],
    regression_summary=regression_summary,
)
results_schema.save_results(results, results_path)
print(f"Saved {results_path}")

## Next step

Open **02-results.ipynb (2/2)** and run it - it automatically finds the most recently
written `results*.json` (this one) and reads the config/data paths back out of it, so
there's nothing to re-enter.